# 21 CV Overlay Videos

Purpose: after `16_deep_cv_yolo_pose_homography.ipynb`, render human-review mp4s with YOLO boxes, tracking ids, pose skeletons, bat lines, and contact-window markers burned onto the original batting clips.

Required input:

- `clips/{FULL_RUN_ID}/clips_v1.parquet`
- at least one of `detections/{FULL_RUN_ID}/detections_v1.parquet` or `pose2d/{FULL_RUN_ID}/pose2d_v1.parquet`

Created outputs:

- `debug/{FULL_RUN_ID}/cv_overlays/{clip_id}_cv_overlay.mp4`
- `debug/{FULL_RUN_ID}/cv_overlay_videos_v1.parquet`
- updated `debug/{FULL_RUN_ID}/debug_overlays_v1.parquet`
- `reports/preflight/cv_overlay_videos_{FULL_RUN_ID}.json`
- `reports/preflight/cv_overlay_videos_{FULL_RUN_ID}_progress.json`


In [ ]:
from pathlib import Path
import os
import sys
import importlib.util


def _mount_drive_if_needed() -> None:
    try:
        from google.colab import drive  # type: ignore
    except ModuleNotFoundError:
        print('Google Colab ではないため Drive mount を skip します。')
        return
    mountpoint = Path('/content/drive')
    mountpoint.mkdir(parents=True, exist_ok=True)
    if (mountpoint / 'MyDrive').exists():
        print('Drive already mounted at /content/drive')
        return
    drive.mount(str(mountpoint))


def _is_repo_dir(path: Path) -> bool:
    return (path / 'src' / 'sport_pipeline' / '__init__.py').exists() and (path / 'configs').exists()


def _resolve_repo_dir() -> Path:
    fixed = Path('/content/drive/MyDrive/codex/batting_codex_handoff')
    candidates = []
    if os.environ.get('BATTING_CODE_ROOT'):
        candidates.append(Path(os.environ['BATTING_CODE_ROOT']))
    candidates.extend([fixed, Path.cwd(), *Path.cwd().parents])
    checked = []
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if str(candidate) in checked:
            continue
        checked.append(str(candidate))
        if _is_repo_dir(candidate):
            return candidate
    raise FileNotFoundError('sport_pipeline repo が見つかりません。BATTING_CODE_ROOT か Drive 配置を確認してください。\n' + '\n'.join(checked))


_mount_drive_if_needed()
REPO_DIR = _resolve_repo_dir()
BASE_DIR = Path('/content/drive/MyDrive/baseball_vision')
CACHE_DIR = Path('/content/cache/baseball_vision')
RUN_PROFILE_NAME = os.environ.get('BASEBALL_VISION_RUN_PROFILE', 'mlb_2024_2026_real_colab_v2.json')
RUN_PROFILE_PATH = REPO_DIR / 'configs' / 'runs' / RUN_PROFILE_NAME

BASE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(REPO_DIR)
src_dir = REPO_DIR / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
if importlib.util.find_spec('sport_pipeline') is None:
    raise ModuleNotFoundError(f'sport_pipeline を import できません: {src_dir}')

from sport_pipeline.pipeline.run_profile import load_run_profile, run_id, stage_settings

RUN_PROFILE = load_run_profile(RUN_PROFILE_PATH)
FULL_RUN_ID = run_id(RUN_PROFILE, 'full_run_id')
OVERLAY_SETTINGS = stage_settings(RUN_PROFILE, 'cv_overlay')

print('REPO_DIR =', REPO_DIR)
print('BASE_DIR =', BASE_DIR)
print('FULL_RUN_ID =', FULL_RUN_ID)
print('OVERLAY_SETTINGS =', OVERLAY_SETTINGS)


In [ ]:
from sport_pipeline.io import read_table

clips_path = BASE_DIR / 'clips' / FULL_RUN_ID / 'clips_v1.parquet'
detections_path = BASE_DIR / 'detections' / FULL_RUN_ID / 'detections_v1.parquet'
pose2d_path = BASE_DIR / 'pose2d' / FULL_RUN_ID / 'pose2d_v1.parquet'
bat_lines_path = BASE_DIR / 'objects' / FULL_RUN_ID / 'bat_line_v1.parquet'

print('clips:', clips_path, clips_path.exists())
print('detections:', detections_path, detections_path.exists())
print('pose2d:', pose2d_path, pose2d_path.exists())
print('bat_lines:', bat_lines_path, bat_lines_path.exists())

if not clips_path.exists():
    raise FileNotFoundError(f'clips_v1 がありません。先に 12_full_cv_preprocess.ipynb を実行してください: {clips_path}')
if not detections_path.exists() and not pose2d_path.exists():
    raise FileNotFoundError('detections / pose2d がありません。先に 16_deep_cv_yolo_pose_homography.ipynb を実行してください。')

clip_rows = read_table(clips_path)
detection_count = len(read_table(detections_path)) if detections_path.exists() else 0
pose_count = len(read_table(pose2d_path)) if pose2d_path.exists() else 0
print('clip rows =', len(clip_rows))
print('detection rows =', detection_count)
print('pose rows =', pose_count)
if detection_count == 0 and pose_count == 0:
    raise RuntimeError('detections / pose2d rows が両方 0 です。16 で ENABLE_YOLO=True、棒人間が必要なら ENABLE_RTMPOSE=True にして pilot してください。')
if pose_count == 0:
    print('WARNING: pose rows が 0 なので、今回の overlay は bbox/track中心で、棒人間 skeleton は出ません。')


In [ ]:
from sport_pipeline.cv.overlay_video import render_cv_overlay_videos

ENABLE_CV_OVERLAY = bool(OVERLAY_SETTINGS.get('execute_default', True))
MAX_CLIPS = OVERLAY_SETTINGS.get('max_clips', 30)
MAX_FRAMES_PER_CLIP = OVERLAY_SETTINGS.get('max_frames_per_clip')
INCLUDE_NO_CV = bool(OVERLAY_SETTINGS.get('include_no_cv', False))
OVERWRITE = bool(OVERLAY_SETTINGS.get('overwrite', False))
UPDATE_CLIPS_MANIFEST = bool(OVERLAY_SETTINGS.get('update_clips_manifest', True))
MIN_POSE_SCORE = float(OVERLAY_SETTINGS.get('min_pose_score', 0.20))
MAX_POSES_PER_FRAME = int(OVERLAY_SETTINGS.get('max_poses_per_frame', 2))

if ENABLE_CV_OVERLAY:
    summary = render_cv_overlay_videos(
        BASE_DIR,
        FULL_RUN_ID,
        max_clips=MAX_CLIPS,
        max_frames_per_clip=MAX_FRAMES_PER_CLIP,
        include_no_cv=INCLUDE_NO_CV,
        overwrite=OVERWRITE,
        update_clips_manifest=UPDATE_CLIPS_MANIFEST,
        min_pose_score=MIN_POSE_SCORE,
        max_poses_per_frame=MAX_POSES_PER_FRAME,
    )
    print('rendered_overlays =', summary['rendered_overlays'])
    print('overlay_dir =', summary['outputs']['overlay_dir'])
    print('overlay_manifest =', summary['outputs']['overlay_manifest'])
    print('summary =', summary['outputs']['summary'])
    print('progress =', summary['outputs']['progress'])
    if summary['errors']:
        print('errors tail =', summary['errors'][-10:])
else:
    print('ENABLE_CV_OVERLAY=False のため overlay mp4 作成は skip しました。')


In [ ]:
from IPython.display import Video, display
from sport_pipeline.io import read_table

overlay_manifest = BASE_DIR / 'debug' / FULL_RUN_ID / 'cv_overlay_videos_v1.parquet'
if overlay_manifest.exists():
    overlay_rows = read_table(overlay_manifest)
    print('overlay rows =', len(overlay_rows))
    for row in overlay_rows[:10]:
        print(row['clip_id'], row['artifact_path'])
    if overlay_rows:
        display(Video(overlay_rows[0]['artifact_path'], embed=False, width=720))
else:
    print('overlay manifest がまだありません:', overlay_manifest)

print('NEXT: 09_report_builder.ipynb を再実行すると、failure browser / clip quality から overlay_path を辿りやすくなります。')
